# Data exploration
Reads `data/` only; all computation happens in `src/`.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.width", 160)

In [2]:
ds = pd.read_parquet("data/processed/dataset.parquet")
ds.groupby(["generator", "mode"]).agg(texts=("id", "size"), median_words=("n_words", "median"))

texts  median_words
generator mode                       
A         multi     406         166.0
          single     83         203.0
B         multi     405         148.0
          single     83         260.0
C         multi     406         215.0
          single     83         276.0
D         multi      80         308.0
E         multi     230         302.5
human     human     406         310.0

## Class and topic distribution of the human sources

In [3]:
src = pd.read_csv("data/sources.csv")
src.groupby(["kind", "topic"]).size().unstack(fill_value=0)

topic,apt,malware,other,phishing,ransomware,vulnerability
kind,,,,,,
advisory,24,2,1,19,31,19
alert,16,6,15,13,15,24
analysis,6,5,0,1,2,4


## One excerpt and its LLM versions (same facts, five generators)

In [4]:
chunk = ds[ds.generator == "D"].chunk_id.iloc[0]
for _, r in ds[(ds.chunk_id == chunk) & ds["mode"].isin(["human", "multi"])].iterrows():
    print(f"--- {r.generator} ({r.n_words} words)\n{r.text[:600]}\n")

--- human (302 words)
The cybersecurity authorities of the United States , Australia , Canada , New Zealand , and the United Kingdom are releasing this joint Cybersecurity Advisory (CSA). The intent of this joint CSA is to warn organizations that Russia's invasion of Ukraine could expose organizations both within and beyond the region to increased malicious cyber activity . This activity may occur as a response to the unprecedented economic costs imposed on Russia as well as materiel support provided by the United States and U.S. allies and partners. Evolving intelligence indicates that the Russian government is ex

--- A (193 words)
Our team has detected a significant increase in malicious cyber activity attributed to the Russian government. The campaign, which appears to be ongoing, targets organizations within and beyond the region, including Ukrainian government and critical infrastructure organizations, as well as countries and organizations providing materiel support to Ukraine. 

## Largest feature differences (Cliff's delta, BH-corrected)

In [5]:
t = pd.read_csv("results/linguistic_tests.csv")
t[t.p_bh < 0.05].sort_values("cliffs_delta", key=abs, ascending=False).head(15)

,generator,feature,llm_median,human_median,cliffs_delta,p,p_bh
102,D,mtld,203.026407,84.082422,0.978571,1.431804e-43,5.206560e-43
134,E,mtld,168.397017,84.082422,0.977254,2.449508e-93,5.598875e-92
132,E,word_len_mean,6.432229,5.728811,0.935479,1.092916e-85,1.942962e-84
103,D,hapax_ratio,0.553228,0.391871,0.930973,1.336886e-39,4.650039e-39
152,E,pos_ADJ,0.119481,0.066895,0.907132,1.095440e-80,1.348234e-79
101,D,ttr,0.684480,0.558260,0.906681,1.193718e-37,3.979060e-37
114,D,rep_unigram,0.315520,0.441740,-0.906681,1.193718e-37,3.979060e-37
115,D,rep_bigram,0.028986,0.094447,-0.893842,1.222624e-36,3.992240e-36
147,E,rep_bigram,0.032931,0.094447,-0.885414,5.858700e-77,6.249280e-76
149,E,pos_PROPN,0.028886,0.114460,-0.876044,2.229989e-75,2.229989e-74


In [6]:
pd.read_csv("results/top_ngrams.csv").groupby(["generator", "side"]).head(8).pivot_table(
    index=["generator", "side"], values="ngram", aggfunc=", ".join)

ngram
generator side                                                   
A         a     our, potential, been, we, measures, to mitigat...
          b     cisa, fbi, see, cisa and, note, cybersecurity,...
B         a     our, potential, mitigate, we, these, to mitiga...
          b     cisa, fbi, see, note, cisa and, the fbi, is a,...
C         a     we, potential, our, measures, threats, vigilan...
          b     cisa, fbi, see, that, used, cisa and, the fbi,...
D         a     anomalous, continuous, monitoring, continuous ...
          b     cisa, cyber, this, fbi, to, cybersecurity, inf...
E         a     these, must, unauthorized, our, we, persistent...
          b     cisa, in, cyber, fbi, used, information, see, ...